# Bike Demo
adapted from https://github.com/brightway-lca/from-the-ground-up/tree/main/try.brightway.dev

In [ ]:
# Imports tell us which libraries we need to run the code.
# The shortnames (like `bc` for `bw2calc`) are just to make it easier to use them.
import bw2calc as bc
import bw2data as bd
import bw2io as bi
import bw2analyzer as ba
import matplotlib.pyplot as plt
import numpy as np
import random
import pandas as pd

## 1. Projects and Database setup

The first thing to learn about `bw2data` is the concept of projects. Each project is self-contained, and independent of other projects. Each has its own subdirectory. This can lead to data duplication, but helps keep each project safe from the changes in the others.

We start in the `default` project:

In [ ]:
bd.projects.current

Let's create a new project:

In [ ]:
bd.projects.set_current(name='demo')
bd.projects.current

To enter our data into BW, we need to create the nodes (or activities), and then the edges (or exchanges). We will create these nodes in a `Database`. A database in BW is just a collection of nodes - it can be large or small, there aren't any general rules.

Let's check what databases we have:

In [ ]:
bd.databases

This should be empty, we haven't created any yet, so let's add to our project. 

We wont be using Ecoinvent in this example, but we can access Biosphere (which is a database full of elementary flow data and is open access) and we can access the LCIA methods so let's add those (this might take a minute)

In [ ]:
bi.remote.install_project(
    "ecoinvent-3.12-biosphere",
    "demo",
    overwrite_existing = True
)

Now let's make a database for our foreground data

In [ ]:
bike_db = bd.Database("bike")
bike_db.register()
bd.databases # now when we check this, we see our new databases

We should make our biosphere database easier to access for later

In [ ]:
bs_db = bd.Database("ecoinvent-3.12-biosphere")

## 2. Activities and Exchanges

Next we can start to create our foreground, in this case, a bike.

We will start with creating our nodes or activities:

In [ ]:
bike = bike_db.new_activity(
    name ='bike',
    unit ='unit',
    location = 'DK',
    type = 'product',
    code = 'bike', # codes must be unique within a database, but can be anything. See what happens if you try to save this activity again.
)
bike.save()

In [ ]:
bike_production = bike_db.new_activity(
    name ='bike production',
    location ='DK',
    type = 'process',
    code = 'bike_production',
)
bike_production.save()

Now try to create the activities for carbon fibre and carbon fibre production  

    we will make it's location 'DE'  
    give the product the code 'cf' and  
    cf_production for the process

In [ ]:
# note to remove the values for these so the students have to write them
cf = bike_db.new_activity(
    name = 'carbon fibre', 
    unit = 'kg',
    location = 'DE',
    type = 'product',
    code = 'cf',
)
cf.save()

cf_production = bike_db.new_activity(
    name = 'carbon fibre production',
    location = 'DE',
    type = 'process',
    code = 'cf_production',
)
cf_production.save()

In our process diagram, we can see that some CO2 is also created during this step. We will include this later. For now, let's make the natural gas activities

In [ ]:
# Try to add your own activities here! 
ng = bike_db.new_activity(
    name = 'natural gas', 
    unit = 'MJ',
    location = 'DE',
    type = 'product',
    code = 'ng',
)
ng.save()

ng_production = bike_db.new_activity(
    name = 'natural gas production',
    location = 'DE',
    type = 'process',
    code = 'ng_production',
)
ng_production.save()

Now we need to connect our activities together using exchanges! 

We'll start from the bike again:

In [ ]:
# This is the production exchange, which links the production of the bike to the bike activity. The amount is usually 1.
bike_production.new_exchange(
    input = bike,
    type = 'production',
    amount = 1,
).save()

# Technosphere exchanges represent how much materials are coming from other processes. 
# In this case, carbon fibre is used in bike production. 
bike_production.new_exchange(
    input = cf,
    type = 'technosphere',
    amount = 2.5,
).save()

# Again we will need a production exchange for the carbon fibre production.
cf_production.new_exchange(
    input = cf,
    type = 'production',
    amount = 1,
).save()

Now try to continue building the exchanges based on the natural gas you have created before.
Ignore the CO2 again for now, we will add that later.

In [ ]:
# Try to make the natural gas exchanges here!
cf_production.new_exchange(
    input = ng,
    type = 'technosphere',
    amount = 237,
).save()

ng_production.new_exchange(
    input = ng,
    type = 'production',
    amount = 1,
).save()

Great job! Now we can have a little look at what we've made:

In [ ]:
# Let's check our activities
for act in bike_db:
    print(act)

In [ ]:
# And the exchanges for our carbon fibre
# You can do this by just printing the exchanges
for exc in cf_production.exchanges():
    print(exc)

In [ ]:
# Or you can use Python's built in list comprehension
[exc for exc in cf_production.exchanges()]

Oh yeah, it's missing our CO2. We should add that next

## 3. Working with larger datasets

First we need to find the CO2 that we want to use in the bioshpere database. Let's look for it using list comprehension.

In [ ]:
# here .lower is used to avoid filtering something just because the capitalization doesn't match
co2 = [flow for flow in bs_db if 'carbon dioxide' in flow['name'].lower()] 
co2

Okay, so there's more than one! Great, we should have known it wouldn't be that simple. Let's have a bit of a closer look at one of these flows 

In [ ]:
# dict() is used to look at the flow as a dictionary and 
# random.choice() is used to select a random element from a list
dict(random.choice(co2))

Now we know what information is attached to these bioshpere flows, we can be more specific about what we're looking for

In [ ]:
co2 = [flow for flow in bs_db if 'carbon dioxide, fossil' == flow['name'].lower() and ('air',) == flow['categories']]
co2

That's better, but it's still giving us a list so we will just say we're taking the first response here

In [ ]:
# in python numbers start from 0 so [0] is used to select the first element of the list
co2 = co2[0]
dict(co2)

Okay, now we need to add it to our carbon fibre production

In [ ]:
cf_production.new_exchange(
    input = co2,
    type = 'biosphere', # this is a biosphere exchange
    amount = 26.6,
).save()

And let's check our exchanges again to make sure!

In [ ]:
[exc for exc in cf_production.exchanges()]

Well done! Our LCI is starting to come together! Next we can find out our impacts!

Note: although we aren't using ecoinvent's technosphere data, it will work the same way that we have seen before. We can find data from ecoinvent using list comprehension and add it to our foreground by using .new_exchange and the type will be 'technosphere'.

## 4. LCIA

If we want to know the impacts of our life cycle, we need to run them based on impact assessment methods (of which there are many). Have a look at how many there are:

In [ ]:
bd.methods

Good thing our trusty list comprehension can help us here again

In [ ]:
[m for m in bd.methods if m[1] == "ReCiPe 2016 v1.03, midpoint (H)"]

let's narrow it down further to just climate change for now

In [ ]:
recipe_CC = [m for m in bd.methods if 'climate change' in m[2].lower() and 'ReCiPe 2016 v1.03, midpoint (H)' == m[1]][0]
recipe_CC

Now we know what method we're using, we can actually figure out the impact of making our bike!

In the next cell, we will execute 4 lines of code to perform our LCA! Let’s get a breakdown of what is going on here:

In [ ]:
# We create an LCA ‘object’ that requires our functional unit and our method. We call our new object lca.
# For the functional unit, we have bike (which we previously defined when we created the bike activity) and we have 1 as the quantity.
# For the method, we have recipe_CC, which we defined in the prior line of code.  
lca = bc.LCA(
    demand = {bike: 1}, 
    method = recipe_CC
) 

# Calculates the life cycle inventory from technosphere and biosphere matrices based on the demand  
lca.lci()

# Calculates the impact by multiplying the characterization matrix from our chosen method with the LCI matrix created in the prior line of code.
lca.lcia()

#Returns the LCA score
lca.score

What if we want to see what is contributing the impacts?

In [ ]:
ca = ba.ContributionAnalysis().annotated_top_processes(lca)

pd.DataFrame([
    {'activity': r[2]['name'], 'score': r[0]}
    for r in ca
])

Cool! We have results!

However, the only thing we've currently connected to the biosphere is CO2 during carbon fibre production which is why our contribution analysis is pretty limited. 

## 5. Exercise

Try adding some elementary flow to your natural gas consumption:

    37 grams of CO2 per MJ  
    0.19 grams of nitrogen oxides per MJ 

And check what that does to the climate change impact of your bike.  
Now that we've added NOx, what other life cycle impact might we look at?  
Can you calculate one other impact for our bike?  

In [ ]:
# Try adding the CO2 to your natural gas production here!
ng_production.new_exchange(
    input = co2,
    type = 'biosphere', # this is a biosphere exchange
    amount = 0.037,
).save()

In [ ]:
# Find the NOx you want to use from the biosphere and add that exchange to your natural gas production here!

nox = [flow for flow in bs_db if 'nitrogen oxides' == flow['name'].lower() and ('air',) == flow['categories']][0]

ng_production.new_exchange(
    input = nox,
    type = 'biosphere', # this is a biosphere exchange
    amount = 0.00019,
).save()

In [ ]:
# Score your LCA using our recipe_CC method here!
lca = bc.LCA(
    demand = {bike: 1}, 
    method = recipe_CC
) 
lca.lci()
lca.lcia()
lca.score

In [ ]:
# Find a different method and score your LCA with that method here!
recipe_tap = [m for m in bd.methods if 'acidification: terrestrial' in m[2].lower() and 'ReCiPe 2016 v1.03, midpoint (H)' == m[1]][0]
recipe_pmfp = [m for m in bd.methods if 'particulate matter formation' in m[2].lower() and 'ReCiPe 2016 v1.03, midpoint (H)' == m[1]][0]
recipe_hofp = [m for m in bd.methods if 'photochemical oxidant formation: human health' in m[2].lower() and 'ReCiPe 2016 v1.03, midpoint (H)' == m[1]][0]
recipe_eofp = [m for m in bd.methods if 'photochemical oxidant formation: terrestrial ecosystems' in m[2].lower() and 'ReCiPe 2016 v1.03, midpoint (H)' == m[1]][0]

lca = bc.LCA(
    demand = {bike: 1}, 
    method = recipe_tap
) 
lca.lci()
lca.lcia()
lca.score

## 6. Uncertainty

Now we know how to navigate our lists and we have made some basic LCA's, we can see some of what BW can do.

Let's look at uncertainty in our bike process. First we need to update our exchanges to include information about their distribution. We can use [stats_arrays](https://stats-arrays.readthedocs.io/en/latest/) to model these.

In [ ]:
# Check our exchanges again to see which ones we want to edit
[exc for exc in cf_production.exchanges()]

Maybe we don't know how much CO2 is emitted during the carbon fibre production

In [ ]:
cf_co2_exc = [exc for exc in cf_production.exchanges()][2]

cf_co2_exc['uncertainty_type'] = 5 # this it a triangular distribution, which is defined by a minimum, maximum and most likely value.
cf_co2_exc['maximum'] = 27.2 
cf_co2_exc['minimum'] = 26
cf_co2_exc.save()

What about if we don't know how much natural gas is used. Let's do triangular distribution with min-max of 150-250 MJ? 

In [ ]:
# Try adding natural gas uncertainty information here!

cf_ng_exc = [exc for exc in cf_production.exchanges()][1]

cf_ng_exc['uncertainty_type'] = 5 
cf_ng_exc['maximum'] = 250 
cf_ng_exc['minimum'] = 150
cf_ng_exc.save()

Now we can score our impacts again! Try running it a couple of times and see what values you get. 

In [ ]:
mc_lca = bc.LCA(
    demand = {bike: 1}, 
    method = recipe_CC,
    use_distributions = True, # This is how we tell BW that we want to use the uncertainty information we just added to our exchanges
) 
mc_lca.lci()
mc_lca.lcia()
mc_lca.score

It's great to see each score individually but really we want to do this many times to see what the distribution is. So let's do that.

In [ ]:
nsims = 10

mc_scores = [mc_lca.score for _ in zip(mc_lca, range(nsims))]

In [ ]:
plt.hist(mc_scores, bins=40)

Okay, it doesn't look like much now but we should aim for more iterations. The more iterations you do, the longer it takes, try 100 and go from there. See how long it takes. 